In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.ablate_cot import make_ablated_prefix_ids, split_qwen3_response

MODEL = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype="auto", device_map="auto"
).eval()


@torch.inference_mode()
def generate(prefix_ids, max_new_tokens=512):
    prefix_ids = torch.tensor([prefix_ids], device=model.device)
    output_ids = model.generate(
        prefix_ids,
        attention_mask=torch.ones_like(prefix_ids),
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(
        output_ids[0, prefix_ids.shape[1] :], skip_special_tokens=True
    )

In [21]:
query = "1 + 1 = ?"
ablated_cot = "1 + 1 = 2"
messages = [{"role": "user", "content": query}]

original_prefix_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, enable_thinking=True
)
original_response = generate(original_prefix_ids)
_, original_answer = split_qwen3_response(original_response)

intervened_prefix_ids = make_ablated_prefix_ids(
    tokenizer, messages, ablated_cot
)
intervened_continuation = generate(intervened_prefix_ids)
intervened_response = (
    f"<think>\n{ablated_cot}\n</think>\n\n{intervened_continuation}"
)
_, intervened_answer = split_qwen3_response(intervened_response)

print("Original answer:", original_answer)
print("Intervened answer:", intervened_answer)
print("Intervened prefix:\n", tokenizer.decode(intervened_prefix_ids))

Original answer: 1 + 1 = 2. 

This is a straightforward addition problem where both numbers are 1. The result is simply the sum of the two numbers. There are no hidden meanings or special operations implied here. 

**Answer:** 2
Intervened answer: 1 + 1 = 2
Intervened prefix:
 <|im_start|>user
1 + 1 = ?<|im_end|>
<|im_start|>assistant
<think>
1 + 1 = 2
</think>




In [26]:
intervened_response

'<think>\n1 + 1 = 2\n</think>\n\n1 + 1 = 2'

In [27]:
# for presentation: 
# first show the 'original cot + response'
# then show the 'ablated cot + response'
# put a .html level visualization of this intervention
print(" <|im_start|>user\n1 + 1 = ?<|im_end|>\n<|im_start|>")
print(original_response)

print("\n--- intervention on CoT ---\n")
print(" <|im_start|>user\n1 + 1 = ?<|im_end|>\n<|im_start|>")
print(intervened_response)

 <|im_start|>user
1 + 1 = ?<|im_end|>
<|im_start|>
<think>
Okay, so I need to figure out what 1 + 1 equals. Let me start by recalling basic arithmetic. When you add two numbers together, you just add their values. In this case, both numbers are 1. So, adding 1 and 1 should give me 2. 

Wait, is there any trick here? Sometimes problems might have hidden meanings or maybe use some special operations. But the question is straightforward. It's just a simple addition problem. There's no indication that it's a more complex operation like multiplication or something else. 

Let me think again. If I have 1 and 1, and I add them, it's like having one object and another one. So, combining them would result in two objects. That makes sense. So, 1 + 1 = 2. 

I don't think there's any other way this could be interpreted. Maybe if there was a different context, like in a different number system or something, but the question is written in base 10, right? So, it's just regular addition. 

Another way



<think>
Okay, so I need to figure out what 1 + 1 equals. Let me start by recalling basic arithmetic. When you add two numbers together, you just add their values. In this case, both numbers are 1. So, adding 1 and 1 should give me 2. 

Wait, is there any trick here? Sometimes problems might have hidden meanings or maybe use some special operations. But the question is straightforward. It's just a simple addition problem. There's no indication that it's a more complex operation like multiplication or something else. 

Let me think again. If I have 1 and 1, and I add them, it's like having one object and another one. So, combining them would result in two objects. That makes sense. So, 1 + 1 = 2. 

I don't think there's any other way this could be interpreted. Maybe if there was a different context, like in a different number system or something, but the question is written in base 10, right? So, it's just regular addition. 

Another way to check is to use the addition table. If I remember correctly, 1 + 1 is 2. So, that's consistent. 

I guess there's no confusion here. The answer should be 2. I don't see any other possible interpretations. Maybe if there was a typo or if the question was part of a larger problem, but as it stands, 1 + 1 equals 2.
</think>

1 + 1 = 2. 

This is a straightforward addition problem where both numbers are 1. The result is simply the sum of the two numbers. There are no hidden meanings or special operations implied here. 

**Answer:** 2